<h2>Description</h2>

L'objectif de ce code est de fusionner toutes les données, donc à savoir les données des champs Elysées avec les données externes. 
Nous voulons donc créer un dataframe qui contient :
<li>les informations météorologiques</li>
<li>les informations sur les vacances et jour fériés</li>
<li>les informations sur l'opération "Paris respire"</li>

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

doc = 'champs_elysees.csv'

print("Version installée de pandas : ")
print(pd.__version__)

Version installée de pandas : 
2.3.0


<h3>Analyse du dataset relatif à l'axe</h3>

In [2]:
df_axe = pd.read_csv("../datasets_axes_bruts/" + doc, sep=";")

# On convertit la date en format convenable 
df_axe['Date et heure de comptage'] = pd.to_datetime(df_axe['Date et heure de comptage'], 
                                      errors='coerce', utc=True).dt.tz_convert('Europe/Paris').dt.tz_localize(None)  

df_axe.head() 

,Identifiant arc,Libelle,Date et heure de comptage,Débit horaire,Taux d'occupation,Etat trafic,Identifiant noeud amont,Libelle noeud amont,Identifiant noeud aval,Libelle noeud aval,Etat arc,Date debut dispo data,Date fin dispo data,geo_point_2d,geo_shape
0,4264,AV_Champs_Elysees,2024-12-09 05:00:00,199.0,2.20945,Fluide,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,Invalide,1996-10-10,2023-01-01,"48.87153587897718, 2.3017227924560624","{""coordinates"": [[2.3009951475338775, 48.87177..."
1,4264,AV_Champs_Elysees,2024-12-09 06:00:00,235.0,2.28778,Fluide,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,Invalide,1996-10-10,2023-01-01,"48.87153587897718, 2.3017227924560624","{""coordinates"": [[2.3009951475338775, 48.87177..."
2,4264,AV_Champs_Elysees,2024-12-09 09:00:00,1041.0,11.63222,Fluide,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,Invalide,1996-10-10,2023-01-01,"48.87153587897718, 2.3017227924560624","{""coordinates"": [[2.3009951475338775, 48.87177..."
3,4264,AV_Champs_Elysees,2025-09-02 09:00:00,1139.0,28.39222,Pré-saturé,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,Ouvert,1996-10-10,2023-01-01,"48.87153587897718, 2.3017227924560624","{""coordinates"": [[2.3009951475338775, 48.87177..."
4,4264,AV_Champs_Elysees,2025-06-05 00:00:00,610.0,9.35833,Fluide,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,Ouvert,1996-10-10,2023-01-01,"48.87153587897718, 2.3017227924560624","{""coordinates"": [[2.3009951475338775, 48.87177..."


In [3]:
df_axe.describe()

,Identifiant arc,Date et heure de comptage,Débit horaire,Taux d'occupation,Identifiant noeud amont,Identifiant noeud aval
count,8723.0,8723,8174.000000,8159.000000,8723.0,8723.0
mean,4264.0,2025-04-25 13:48:04.374641664,738.376070,15.297486,2294.0,2293.0
min,4264.0,2024-10-01 05:00:00,0.000000,0.000000,2294.0,2293.0
25%,4264.0,2025-01-22 10:30:00,532.000000,7.377780,2294.0,2293.0
50%,4264.0,2025-04-25 08:00:00,807.000000,15.156670,2294.0,2293.0
75%,4264.0,2025-08-06 03:30:00,944.000000,21.566945,2294.0,2293.0
max,4264.0,2025-11-06 00:00:00,2190.000000,77.545560,2294.0,2293.0
std,0.0,NaN,286.144772,9.222341,0.0,0.0


In [4]:
df_axe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8723 entries, 0 to 8722
Data columns (total 15 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Identifiant arc            8723 non-null   int64         
 1   Libelle                    8723 non-null   object        
 2   Date et heure de comptage  8723 non-null   datetime64[ns]
 3   Débit horaire              8174 non-null   float64       
 4   Taux d'occupation          8159 non-null   float64       
 5   Etat trafic                8723 non-null   object        
 6   Identifiant noeud amont    8723 non-null   int64         
 7   Libelle noeud amont        8723 non-null   object        
 8   Identifiant noeud aval     8723 non-null   int64         
 9   Libelle noeud aval         8723 non-null   object        
 10  Etat arc                   8723 non-null   object        
 11  Date debut dispo data      8723 non-null   object        
 12  Date f

In [5]:
df_axe['date'] = df_axe['Date et heure de comptage'].dt.date
df_axe['heure'] = df_axe['Date et heure de comptage'].dt.hour
df_axe['dow'] = df_axe['Date et heure de comptage'].dt.dayofweek
df_axe['mois'] = df_axe['Date et heure de comptage'].dt.month
df_axe['annee'] = df_axe['Date et heure de comptage'].dt.year
df_axe['jour_mois'] = df_axe['Date et heure de comptage'].dt.day
jour_map = {0:'Lun',1:'Mar',2:'Mer',3:'Jeu',4:'Ven',5:'Sam',6:'Dim'}
df_axe['jour_semaine'] = df_axe['dow'].map(jour_map)

In [6]:
fig = px.line(
    df_axe.sort_values('Date et heure de comptage'),
    x='Date et heure de comptage', y='Débit horaire',
    title=f"Débit horaire",
    labels={'Date et heure de comptage':'Date/Heure','Débit horaire':'Débit (véh/h)'}
)

fig.show()

In [7]:
profil = (df_axe
          .groupby(['jour_semaine','heure'], as_index=False)['Débit horaire']
          .mean())

fig = px.line(
    profil, x='heure', y='Débit horaire', color='jour_semaine',
    markers=True, title=f"Profil horaire moyen par jour",
    labels={'heure':'Heure','Débit horaire':'Débit moyen (véh/h)','jour_semaine':'Jour'}
)
fig.update_layout(template='simple_white')
fig.show()

In [8]:
heat = (df_axe
        .groupby(['jour_semaine','heure'], as_index=False)['Débit horaire']
        .mean())


heat['jour_semaine'] = pd.Categorical(heat['jour_semaine'],
                                      categories=['Lun','Mar','Mer','Jeu','Ven','Sam','Dim'],
                                      ordered=True)

fig = px.imshow(
    heat.pivot(index='jour_semaine', columns='heure', values='Débit horaire').values,
    labels=dict(x="Heure", y="Jour", color="Débit moyen"),
    x=list(range(24)),
    y=['Lun','Mar','Mer','Jeu','Ven','Sam','Dim'],
    title=f"Carte de chaleur"
)
fig.update_layout(template='simple_white')
fig.show()


In [9]:
df_scatter = df_axe.dropna(subset=['Débit horaire','Taux d\'occupation']).copy()
fig = px.scatter(
    df_scatter, x='Taux d\'occupation', y='Débit horaire',
    color='jour_semaine', opacity=0.7,
    title=f"Débit vs Taux d’occupation",
    labels={'Taux d\'occupation':'Taux d’occupation','Débit horaire':'Débit (véh/h)'}
)

fig.update_layout(template='simple_white')
fig.show()

In [10]:
box = df_axe.dropna(subset=['Débit horaire']).copy()
fig = px.box(
    box, x='jour_semaine', y='Débit horaire', points='all',
    title=f"Distribution du débit par jour",
    labels={'jour_semaine':'Jour','Débit horaire':'Débit (véh/h)'}
)
fig.update_layout(template='simple_white')
fig.show()


In [11]:
etat_jour = (df_axe
             .assign(jour=pd.to_datetime(df_axe['date']))
             .groupby(['jour','Etat trafic'], as_index=False)
             .size())

fig = px.area(
    etat_jour, x='jour', y='size', color='Etat trafic',
    title=f"État du trafic (comptes journaliers)",
    labels={'jour':'Date','size':'Occurrences'}
)
fig.update_layout(template='simple_white', hovermode='x unified')
fig.show()

In [12]:
mensuel = (df_axe
           .set_index('Date et heure de comptage')
           .resample('MS')['Débit horaire']
           .mean()
           .to_frame('debit_moyen')
           .reset_index())

mensuel['trend_3m'] = mensuel['debit_moyen'].rolling(3, min_periods=1).mean()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=mensuel['Date et heure de comptage'], y=mensuel['debit_moyen'],
    mode='lines+markers', name='Moyenne mensuelle'
))
fig.add_trace(go.Scatter(
    x=mensuel['Date et heure de comptage'], y=mensuel['trend_3m'],
    mode='lines', name='Tendance (MM-3)', line=dict(width=4)
))

fig.update_layout(
    title="Débit horaire – tendance mensuelle (moyenne + lissage 3 mois)",
    xaxis_title="Mois",
    yaxis_title="Débit moyen (véh/h)",
    template="simple_white",
    hovermode="x unified"
)
fig.show()


In [13]:
profil = (df_axe
          .groupby(['annee','mois'], as_index=False)['Débit horaire']
          .mean()
          .rename(columns={'Débit horaire':'debit_moyen'}))

fig = px.line(
    profil, x='mois', y='debit_moyen', color='annee',
    markers=True,
    title="Profil mensuel du débit par année",
    labels={'mois':'Mois', 'debit_moyen':'Débit moyen (véh/h)', 'annee':'Année'}
)
fig.update_layout(template='simple_white', xaxis=dict(dtick=1))
fig.show()


In [14]:
df_axe['heure_sin'] = np.sin(2 * np.pi * df_axe['heure'] / 24)
df_axe['heure_cos'] = np.cos(2 * np.pi * df_axe['heure'] / 24)

df_axe['jour_sin'] = np.sin(2 * np.pi * df_axe['dow'] / 7)
df_axe['jour_cos'] = np.cos(2 * np.pi * df_axe['dow'] / 7)

df_axe['mois_sin'] = np.sin(2 * np.pi * df_axe['mois'] / 12)
df_axe['mois_cos'] = np.cos(2 * np.pi * df_axe['mois'] / 12)

In [15]:
df_axe.head(10)

,Identifiant arc,Libelle,Date et heure de comptage,Débit horaire,Taux d'occupation,Etat trafic,Identifiant noeud amont,Libelle noeud amont,Identifiant noeud aval,Libelle noeud aval,...,mois,annee,jour_mois,jour_semaine,heure_sin,heure_cos,jour_sin,jour_cos,mois_sin,mois_cos
0,4264,AV_Champs_Elysees,2024-12-09 05:00:00,199.0,2.20945,Fluide,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,...,12,2024,9,Lun,0.965926,2.588190e-01,0.000000,1.000000,-2.449294e-16,1.000000e+00
1,4264,AV_Champs_Elysees,2024-12-09 06:00:00,235.0,2.28778,Fluide,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,...,12,2024,9,Lun,1.000000,6.123234e-17,0.000000,1.000000,-2.449294e-16,1.000000e+00
2,4264,AV_Champs_Elysees,2024-12-09 09:00:00,1041.0,11.63222,Fluide,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,...,12,2024,9,Lun,0.707107,-7.071068e-01,0.000000,1.000000,-2.449294e-16,1.000000e+00
3,4264,AV_Champs_Elysees,2025-09-02 09:00:00,1139.0,28.39222,Pré-saturé,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,...,9,2025,2,Mar,0.707107,-7.071068e-01,0.781831,0.623490,-1.000000e+00,-1.836970e-16
4,4264,AV_Champs_Elysees,2025-06-05 00:00:00,610.0,9.35833,Fluide,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,...,6,2025,5,Jeu,0.000000,1.000000e+00,0.433884,-0.900969,1.224647e-16,-1.000000e+00
5,4264,AV_Champs_Elysees,2025-06-04 23:00:00,618.0,11.21278,Fluide,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,...,6,2025,4,Mer,-0.258819,9.659258e-01,0.974928,-0.222521,1.224647e-16,-1.000000e+00
6,4264,AV_Champs_Elysees,2025-06-04 21:00:00,825.0,17.61167,Pré-saturé,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,...,6,2025,4,Mer,-0.707107,7.071068e-01,0.974928,-0.222521,1.224647e-16,-1.000000e+00
7,4264,AV_Champs_Elysees,2025-06-04 20:00:00,1028.0,23.76667,Pré-saturé,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,...,6,2025,4,Mer,-0.866025,5.000000e-01,0.974928,-0.222521,1.224647e-16,-1.000000e+00
8,4264,AV_Champs_Elysees,2025-06-04 17:00:00,937.0,21.89167,Pré-saturé,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,...,6,2025,4,Mer,-0.965926,-2.588190e-01,0.974928,-0.222521,1.224647e-16,-1.000000e+00
9,4264,AV_Champs_Elysees,2025-04-03 11:00:00,1138.0,16.94889,Pré-saturé,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,...,4,2025,3,Jeu,0.258819,-9.659258e-01,0.433884,-0.900969,8.660254e-01,-5.000000e-01


<h3>Nous insérons d'abord les données météorologiques</h3>

In [16]:
meteo = pd.read_csv("../datasets_externes_clean/meteo.csv", sep=";")
meteo['datetime'] = pd.to_datetime(meteo['datetime'])

In [17]:
df_merge = pd.merge(df_axe, meteo, left_on="Date et heure de comptage", right_on="datetime", how="left")

In [18]:
df_merge.describe()

,Identifiant arc,Date et heure de comptage,Débit horaire,Taux d'occupation,Identifiant noeud amont,Identifiant noeud aval,heure,dow,mois,annee,...,jour_cos,mois_sin,mois_cos,Unnamed: 0,precipitations heure,duree prec (en min),force moyenne vent (m/s),Température,ensoleillement (en min),datetime
count,8723.0,8723,8174.000000,8159.000000,8723.0,8723.0,8723.000000,8723.000000,8723.000000,8723.000000,...,8723.000000,8.723000e+03,8.723000e+03,8558.000000,8558.000000,8558.000000,8558.000000,8558.000000,8558.000000,8558
mean,4264.0,2025-04-25 13:48:04.374641664,738.376070,15.297486,2294.0,2293.0,11.506592,2.994612,6.656082,2024.803508,...,-0.003777,-5.378095e-02,2.876877e-02,11445.416920,0.078768,3.912713,2.932344,13.242487,13.199112,2025-04-21 21:25:00.911428096
min,4264.0,2024-10-01 05:00:00,0.000000,0.000000,2294.0,2293.0,0.000000,0.000000,1.000000,2024.000000,...,-0.900969,-1.000000e+00,-1.000000e+00,6581.000000,0.000000,0.000000,0.000000,-3.600000,0.000000,2024-10-01 05:00:00
25%,4264.0,2025-01-22 10:30:00,532.000000,7.377780,2294.0,2293.0,6.000000,1.000000,4.000000,2025.000000,...,-0.900969,-8.660254e-01,-5.000000e-01,9257.250000,0.000000,0.000000,2.000000,8.600000,0.000000,2025-01-20 17:15:00
50%,4264.0,2025-04-25 08:00:00,807.000000,15.156670,2294.0,2293.0,12.000000,3.000000,7.000000,2025.000000,...,-0.222521,-2.449294e-16,6.123234e-17,11445.500000,0.000000,0.000000,2.700000,13.000000,0.000000,2025-04-21 21:30:00
75%,4264.0,2025-08-06 03:30:00,944.000000,21.566945,2294.0,2293.0,17.000000,5.000000,10.000000,2025.000000,...,0.623490,5.000000e-01,5.000000e-01,13871.750000,0.000000,0.000000,3.700000,17.800000,21.000000,2025-07-31 23:45:00
max,4264.0,2025-11-06 00:00:00,2190.000000,77.545560,2294.0,2293.0,23.000000,6.000000,12.000000,2025.000000,...,1.000000,1.000000e+00,1.000000e+00,16035.000000,14.300000,60.000000,9.300000,37.600000,60.000000,2025-10-30 03:00:00
std,0.0,NaN,286.144772,9.222341,0.0,0.0,6.919745,1.996062,3.431450,0.397368,...,0.707323,7.319658e-01,6.786902e-01,2713.789082,0.523691,12.817157,1.319631,6.740669,22.053415,NaN


In [19]:
df_merge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8723 entries, 0 to 8722
Data columns (total 36 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Identifiant arc            8723 non-null   int64         
 1   Libelle                    8723 non-null   object        
 2   Date et heure de comptage  8723 non-null   datetime64[ns]
 3   Débit horaire              8174 non-null   float64       
 4   Taux d'occupation          8159 non-null   float64       
 5   Etat trafic                8723 non-null   object        
 6   Identifiant noeud amont    8723 non-null   int64         
 7   Libelle noeud amont        8723 non-null   object        
 8   Identifiant noeud aval     8723 non-null   int64         
 9   Libelle noeud aval         8723 non-null   object        
 10  Etat arc                   8723 non-null   object        
 11  Date debut dispo data      8723 non-null   object        
 12  Date f

In [20]:
df_axe = df_merge

<h3>Nous allons maintenant ajouter les données relatives au jours piétonnisés</h3>

Nous allons donc ajouter une colonne de 1 ou 0 pour indiquer si oui ou non il s'agissait d'un jour piéton.

In [21]:
pieton = pd.read_csv("../datasets_externes_clean/pieton.csv", sep=";")
pieton.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  20 non-null     int64 
 1   date_debut  20 non-null     object
 2   date_fin    20 non-null     object
dtypes: int64(1), object(2)
memory usage: 608.0+ bytes


In [22]:
pieton['date_debut'] = pd.to_datetime(pieton['date_debut'])
pieton['date_fin'] = pd.to_datetime(pieton['date_fin'])

Nous écrivons ci-dessous une fonction pour déterminer si pour une date donnée, certains axes parisiens étaient piétonnisés.

In [23]:
def est_pietonnise(date):
    debut = pieton['date_debut']
    fin = pieton['date_fin']
    for i in range (len(debut)):
        if date >= debut[i] and date < fin[i]:
            return 1
    return 0

df_axe['est_pieton'] = df_axe['Date et heure de comptage'].apply(est_pietonnise)

In [24]:
df_axe[df_axe['est_pieton'] == 1][['Date et heure de comptage', 'est_pieton']]

,Date et heure de comptage,est_pieton
80,2024-11-03 16:00:00,1
81,2024-11-03 15:00:00,1
82,2024-11-03 12:00:00,1
83,2024-11-03 11:00:00,1
84,2024-11-03 10:00:00,1
...,...,...
6719,2025-09-21 16:00:00,1
6720,2025-09-21 14:00:00,1
6721,2025-09-21 10:00:00,1
6722,2025-09-21 08:00:00,1


In [25]:
df_axe = df_axe.sort_values(by="Date et heure de comptage", ascending=True)

<h3>On va maintenant ajouter des données sur les vacances, jours fériés...</h3>

In [26]:
vacances = pd.read_csv("../datasets_externes_clean/vacances.csv", sep=";")
vacances ['Date de début'] = pd.to_datetime(vacances['Date de début'])
vacances ['Date de fin'] = pd.to_datetime(vacances['Date de fin'])

In [27]:
def est_jour_vacances(dat):
    date_sans_heure = dat.date()
    debut = vacances['Date de début'].dt.date
    fin = vacances['Date de fin'].dt.date
    for i in range (len(debut)):
        if date_sans_heure >= debut[i] and date_sans_heure < fin[i]:
            return 1
    return 0

df_axe['est_vacances'] = df_axe['Date et heure de comptage'].apply(est_jour_vacances)

In [28]:
df = df_axe[df_axe['est_vacances'] == 1][['Date et heure de comptage', 'est_vacances']]

In [29]:
df.head(50)

,Date et heure de comptage,est_vacances
4772,2024-10-19 00:00:00,1
6410,2024-10-19 01:00:00,1
6409,2024-10-19 02:00:00,1
4937,2024-10-19 03:00:00,1
4936,2024-10-19 04:00:00,1
6408,2024-10-19 05:00:00,1
4935,2024-10-19 06:00:00,1
4934,2024-10-19 07:00:00,1
4933,2024-10-19 08:00:00,1
6407,2024-10-19 09:00:00,1


In [30]:
def est_avant_vacances(date):
    date_sans_heure = date.date()
    debut = vacances['Date de début'].dt.date
    for i in range (len(debut)):
        if date_sans_heure == debut[i] - pd.Timedelta(days=1):
            return 1
    return 0

df_axe['est_avant_vacances'] = df_axe['Date et heure de comptage'].apply(est_avant_vacances)

In [31]:
df_axe[df_axe['est_avant_vacances'] == 1][['est_avant_vacances', 'Date et heure de comptage']]

,est_avant_vacances,Date et heure de comptage
6137,1,2024-10-18 00:00:00
5939,1,2024-10-18 01:00:00
5938,1,2024-10-18 02:00:00
6114,1,2024-10-18 03:00:00
5937,1,2024-10-18 04:00:00
...,...,...
4273,1,2025-10-17 19:00:00
4274,1,2025-10-17 20:00:00
3993,1,2025-10-17 21:00:00
3994,1,2025-10-17 22:00:00


In [32]:
ferie = pd.read_csv("../datasets_externes_clean/ferie.csv", sep=";")

ferie['Date de début'] = pd.to_datetime(ferie['Date de début'], format='%d/%m/%Y')

In [33]:
def est_ferie(date):
    date = date.date()
    j_ferie = ferie['Date de début'].dt.date
    for el in j_ferie :
        if date == el:
            return 1
    return 0

df_axe['est_ferie'] = df_axe['Date et heure de comptage'].apply(est_ferie)

In [34]:
df_axe[df_axe['est_ferie']==1]['Date et heure de comptage']

8706   2024-11-01 00:00:00
7918   2024-11-01 01:00:00
8523   2024-11-01 02:00:00
8522   2024-11-01 03:00:00
2793   2024-11-01 04:00:00
               ...        
648    2025-11-01 19:00:00
605    2025-11-01 20:00:00
649    2025-11-01 21:00:00
606    2025-11-01 22:00:00
650    2025-11-01 23:00:00
Name: Date et heure de comptage, Length: 264, dtype: datetime64[ns]

In [35]:
def est_avant_ferie(date):
    date = date.date()
    j_ferie = ferie['Date de début'].dt.date
    for el in j_ferie :
        if date == el - pd.Timedelta(days=1):
            return 1
    return 0

df_axe['est_avant_ferie'] = df_axe['Date et heure de comptage'].apply(est_avant_ferie)

In [36]:
df_axe[df_axe['est_avant_ferie']==1]['Date et heure de comptage']

8597   2024-10-31 00:00:00
8722   2024-10-31 01:00:00
8721   2024-10-31 02:00:00
8699   2024-10-31 03:00:00
8698   2024-10-31 04:00:00
               ...        
8702   2025-10-31 19:00:00
8701   2025-10-31 20:00:00
8559   2025-10-31 21:00:00
8558   2025-10-31 22:00:00
8557   2025-10-31 23:00:00
Name: Date et heure de comptage, Length: 264, dtype: datetime64[ns]

In [37]:
df_axe['est_weekend'] = df_axe['dow'].isin([5, 6]).astype(int)

Nous vérifions maintenant la composition du dataset

In [38]:
df_axe = df_axe.drop('Unnamed: 0', axis = 1)
df_axe.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8723 entries, 1072 to 1574
Data columns (total 41 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Identifiant arc            8723 non-null   int64         
 1   Libelle                    8723 non-null   object        
 2   Date et heure de comptage  8723 non-null   datetime64[ns]
 3   Débit horaire              8174 non-null   float64       
 4   Taux d'occupation          8159 non-null   float64       
 5   Etat trafic                8723 non-null   object        
 6   Identifiant noeud amont    8723 non-null   int64         
 7   Libelle noeud amont        8723 non-null   object        
 8   Identifiant noeud aval     8723 non-null   int64         
 9   Libelle noeud aval         8723 non-null   object        
 10  Etat arc                   8723 non-null   object        
 11  Date debut dispo data      8723 non-null   object        
 12  Date fin

In [39]:
df_axe.describe()

,Identifiant arc,Date et heure de comptage,Débit horaire,Taux d'occupation,Identifiant noeud amont,Identifiant noeud aval,heure,dow,mois,annee,...,force moyenne vent (m/s),Température,ensoleillement (en min),datetime,est_pieton,est_vacances,est_avant_vacances,est_ferie,est_avant_ferie,est_weekend
count,8723.0,8723,8174.000000,8159.000000,8723.0,8723.0,8723.000000,8723.000000,8723.000000,8723.000000,...,8558.000000,8558.000000,8558.000000,8558,8723.000000,8723.000000,8723.000000,8723.000000,8723.000000,8723.000000
mean,4264.0,2025-04-25 13:48:04.374641920,738.376070,15.297486,2294.0,2293.0,11.506592,2.994612,6.656082,2024.803508,...,2.932344,13.242487,13.199112,2025-04-21 21:25:00.911428096,0.012610,0.357560,0.016623,0.030265,0.030265,0.283159
min,4264.0,2024-10-01 05:00:00,0.000000,0.000000,2294.0,2293.0,0.000000,0.000000,1.000000,2024.000000,...,0.000000,-3.600000,0.000000,2024-10-01 05:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,4264.0,2025-01-22 10:30:00,532.000000,7.377780,2294.0,2293.0,6.000000,1.000000,4.000000,2025.000000,...,2.000000,8.600000,0.000000,2025-01-20 17:15:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,4264.0,2025-04-25 08:00:00,807.000000,15.156670,2294.0,2293.0,12.000000,3.000000,7.000000,2025.000000,...,2.700000,13.000000,0.000000,2025-04-21 21:30:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,4264.0,2025-08-06 03:30:00,944.000000,21.566945,2294.0,2293.0,17.000000,5.000000,10.000000,2025.000000,...,3.700000,17.800000,21.000000,2025-07-31 23:45:00,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
max,4264.0,2025-11-06 00:00:00,2190.000000,77.545560,2294.0,2293.0,23.000000,6.000000,12.000000,2025.000000,...,9.300000,37.600000,60.000000,2025-10-30 03:00:00,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
std,0.0,NaN,286.144772,9.222341,0.0,0.0,6.919745,1.996062,3.431450,0.397368,...,1.319631,6.740669,22.053415,NaN,0.111592,0.479309,0.127860,0.171325,0.171325,0.450559


In [40]:
df_axe.to_csv("../datasets_axes_with_all_features/" + doc , sep=";")